# HiCache Vanilla vs Custom Benchmark

This notebook is for Google Colab. It clones your branch, installs SGLang, runs HiCache in `vanilla` and `custom` modes, benchmarks both with the HiCache benchmark, and saves outputs in `benchmark/hicache/results`.

In [ ]:
BRANCH = "test-ramcharan"
REPO_URL = "https://github.com/<your-username-or-org>/sglang.git"  # replace this

MODEL_PATH = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 30000
HICACHE_RATIO = 1.2
MEM_FRACTION_STATIC = 0.70
REQUEST_RATE = 4
NUM_PROMPTS = 20

REPO_DIR = "/content/sglang"
BENCH_DIR = f"{REPO_DIR}/benchmark/hicache"
RESULTS_DIR = f"{BENCH_DIR}/results"

## Clone The Branch

In [ ]:
%%bash
set -e
cd /content
rm -rf sglang
git clone -b test-ramcharan "$REPO_URL" sglang
cd sglang
git branch --show-current

## Install SGLang

In [ ]:
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python || true
python -m pip install -U pip setuptools wheel
python -m pip install -e python

## Optional Hugging Face Login

Run this only if the model you use requires authentication.

In [ ]:
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")

## Download HiCache Benchmark Dataset

In [ ]:
%%bash
set -e
cd /content/sglang/benchmark/hicache
chmod +x download.sh
./download.sh loogle
mkdir -p results
ls -lh

## Server Helpers

In [ ]:
import os
import subprocess
import time
import requests

server_proc = None

def stop_server():
    global server_proc
    if server_proc is not None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=20)
        except Exception:
            server_proc.kill()
        server_proc = None
    os.system("pkill -f 'sglang.launch_server' || true")

def start_server(hicache_impl: str, hicache_backup_policy: str):
    global server_proc
    stop_server()
    os.makedirs(RESULTS_DIR, exist_ok=True)
    log_file = f"{RESULTS_DIR}/{hicache_impl}_{hicache_backup_policy}_server.log"
    cmd = f'''cd {REPO_DIR} && python3 -m sglang.launch_server \
      --model-path {MODEL_PATH} \
      --port {PORT} \
      --enable-hierarchical-cache \
      --hicache-impl {hicache_impl} \
      --hicache-backup-policy {hicache_backup_policy} \
      --hicache-ratio {HICACHE_RATIO} \
      --mem-fraction-static {MEM_FRACTION_STATIC}'''
    with open(log_file, "w") as f:
        server_proc = subprocess.Popen(
            cmd,
            shell=True,
            executable="/bin/bash",
            stdout=f,
            stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,
        )

    base_url = f"http://127.0.0.1:{PORT}"
    for _ in range(180):
        try:
            resp = requests.get(base_url + "/get_server_info", timeout=2)
            if resp.status_code == 200:
                print(f"{hicache_impl}/{hicache_backup_policy} server is ready")
                return
        except Exception:
            pass
        time.sleep(2)

    raise RuntimeError(f"{hicache_impl} server did not start. Check {log_file}")

## Start Vanilla HiCache Server

In [ ]:
start_server("vanilla", "fixed")

## Benchmark Vanilla HiCache With Multiturn

In [ ]:
%%bash
set -e
cd /content/sglang/benchmark/hicache

python3 bench_serving.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --backend sglang \
  --dataset-path longdep_qa.json \
  --dataset-name loogle \
  --request-rate 4 \
  --num-prompts 20 \
  --port 30000 \
  --enable-multiturn \
  --disable-shuffle \
  | tee results/vanilla_multiturn.log

## Start Custom HiCache Server

In [ ]:
start_server("custom", "fixed")

## Benchmark Custom HiCache With Multiturn

In [ ]:
%%bash
set -e
cd /content/sglang/benchmark/hicache

python3 bench_serving.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --backend sglang \
  --dataset-path longdep_qa.json \
  --dataset-name loogle \
  --request-rate 4 \
  --num-prompts 20 \
  --port 30000 \
  --enable-multiturn \
  --disable-shuffle \
  | tee results/custom_multiturn.log

## Optional Shared-Prefix Run

If you have time, run both shared-prefix cases too.

In [ ]:
start_server("vanilla", "fixed")

In [ ]:
%%bash
set -e
cd /content/sglang/benchmark/hicache

python3 bench_serving.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --backend sglang \
  --dataset-path longdep_qa.json \
  --dataset-name loogle \
  --request-rate 4 \
  --num-prompts 20 \
  --port 30000 \
  --enable-shared-prefix \
  --disable-shuffle \
  | tee results/vanilla_shared_prefix.log

In [ ]:
start_server("custom", "adaptive")

In [ ]:
%%bash
set -e
cd /content/sglang/benchmark/hicache

python3 bench_serving.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --backend sglang \
  --dataset-path longdep_qa.json \
  --dataset-name loogle \
  --request-rate 4 \
  --num-prompts 20 \
  --port 30000 \
  --enable-shared-prefix \
  --disable-shuffle \
  | tee results/custom_shared_prefix.log

## Inspect Saved Results

In [ ]:
%%bash
cd /content/sglang/benchmark/hicache/results
ls -lh

## Stop Server

In [ ]:
stop_server()
print("Done")